# LibriSpeech Speech Recognition Pipeline: Conformer with Connectionist Temporal Classification

This notebook implements an end-to-end Automatic Speech Recognition (ASR) pipeline using a Conformer (Convolution-augmented Transformer) encoder and Connectionist Temporal Classification (CTC) loss on the LibriSpeech dataset (`torchaudio.datasets.LIBRISPEECH`).

The code is organized into sequential functional stages:
1. Environment setup and dependency installation
2. Telemetry dashboard extension
3. Core library imports and environment setup
4. Compute device allocation
5. Hyperparameter and directory configuration
6. Character tokenization and text preprocessing
7. Acoustic feature extraction and SpecAugment transformation
8. Dataset ingestion and DataLoader instantiation
9. Exploratory speech waveform and spectrogram inspection
10. Neural network architecture definition
11. Model instantiation and parameter summary
12. CTC decoding and error rate metrics
13. Loss function, optimizer, and training loop execution
14. Interactive speech recognition inference

## 1. Environment Setup and Dependency Installation

Install all required runtime dependencies including core deep learning libraries, accelerator backends, telemetry tooling, and speech recognition evaluation metrics.

In [ ]:
%pip install -q torch torchvision torchaudio torch_xla tensorboard matplotlib pandas ipywidgets scipy torchmetrics safetensors

## 2. Telemetry Dashboard Extension

Load the inline telemetry viewer to monitor training progression, validation CTC loss, CER, and WER directly within the execution environment.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ../../runs

## 3. Core Library Imports and Environment Setup

Import numerical computing, audio transforms, neural network layers, accelerator runtime interfaces, and tokenization utilities. Set deterministic random seeds.

In [ ]:
import json
import os
import math
import random
import re
import zipfile
from datetime import datetime, timezone, timedelta
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
import torchaudio
from torchaudio.datasets import LIBRISPEECH
import torchaudio.transforms as T
import IPython.display as ipd
from safetensors.torch import save as safetensors_save, load as safetensors_load

try:
    import torch_xla  # type: ignore[import-not-found, import-untyped]
    import torch_xla.core.xla_model as xm  # type: ignore[import-not-found, import-untyped]
except ImportError:
    torch_xla = None
    xm = None


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    if xm is not None:
        try:
            xm.set_rng_state(seed)
        except Exception:
            pass


seed_everything(42)
print(f"PyTorch: {torch.__version__} | TorchAudio: {torchaudio.__version__}")


def save_model_zip(
    model: torch.nn.Module,
    metadata: dict,
    save_path: Path,
    loss_fn: torch.nn.Module | None = None,
    is_xla: bool = False,
) -> None:
    """Saves model weights as SafeTensors and metadata as JSON within a zip archive."""
    save_path.parent.mkdir(parents=True, exist_ok=True)
    state_dict = {
        k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()
    }
    tensor_bytes = safetensors_save(state_dict)

    metadata_clean = {}
    for k, v in metadata.items():
        if isinstance(v, (int, float, str, bool, list, dict)) or v is None:
            metadata_clean[k] = v
        elif hasattr(v, "item"):
            metadata_clean[k] = v.item()
        elif hasattr(v, "tolist"):
            metadata_clean[k] = v.tolist()
        else:
            metadata_clean[k] = str(v)

    meta_bytes = json.dumps(metadata_clean, indent=2).encode("utf-8")

    with zipfile.ZipFile(save_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("model.safetensors", tensor_bytes)
        zf.writestr("metadata.json", meta_bytes)
        if loss_fn is not None and len(loss_fn.state_dict()) > 0:
            loss_dict = {
                k: v.detach().cpu().contiguous()
                for k, v in loss_fn.state_dict().items()
            }
            zf.writestr("loss_fn.safetensors", safetensors_save(loss_dict))


def load_model_zip(
    model: torch.nn.Module,
    load_path: Path,
    device: torch.device,
    loss_fn: torch.nn.Module | None = None,
) -> dict:
    """Loads model weights from SafeTensors inside zip archive and returns metadata dictionary."""
    with zipfile.ZipFile(load_path, "r") as zf:
        tensor_bytes = zf.read("model.safetensors")
        meta_bytes = zf.read("metadata.json")
        if loss_fn is not None and "loss_fn.safetensors" in zf.namelist():
            loss_bytes = zf.read("loss_fn.safetensors")
            loss_dict = safetensors_load(loss_bytes)
            loss_dict = {k: v.to(device) for k, v in loss_dict.items()}
            loss_fn.load_state_dict(loss_dict)

    state_dict = safetensors_load(tensor_bytes)
    state_dict = {k: v.to(device) for k, v in state_dict.items()}
    model.load_state_dict(state_dict)

    metadata = json.loads(meta_bytes.decode("utf-8"))
    return metadata

## 4. Compute Device Allocation

Initialize and allocate the compute accelerator (Cloud TPU via PyTorch XLA, NVIDIA CUDA GPU, Apple Silicon MPS, or CPU).

In [ ]:
device = None
device_name = None

# 1. Prioritize Cloud TPU
if xm is not None:
    try:
        xla_dev = xm.xla_device()
        if xm.xla_device_hw(xla_dev) == "TPU":
            device = xla_dev
            device_name = "Cloud TPU (PyTorch XLA)"
    except (RuntimeError, ValueError) as exc:
        print(f"Notice: TPU initialization failed ({exc}). Falling back to GPU/CPU.")

# 2. Prioritize GPU (NVIDIA CUDA or Apple Silicon MPS)
if device is None and torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = f"NVIDIA GPU ({torch.cuda.get_device_name(0)})"
elif (
    device is None
    and hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    device = torch.device("mps")
    device_name = "Apple Silicon GPU (MPS)"

# 3. Fallback to CPU
if device is None:
    device = torch.device("cpu")
    device_name = "CPU"

is_xla = device.type == "xla"

print(f"Allocated Compute Device: {device} [{device_name}]")

## 5. Hyperparameter and Directory Configuration

Define file system paths for data caching, telemetry logs, and model checkpoints. Set acoustic parameters ($16\text{ kHz}$, $80$ Mel bins), Conformer dimensions ($d_{\text{model}}=144, 4\text{ layers}$), and optimization hyperparameters.

In [ ]:
# ── Directory Paths ────────────────────────────────────────────────────────
DATA_DIR = Path("../../data").resolve()
RUN_TIMESTAMP = datetime.now(tz=timezone(timedelta(hours=1))).strftime(
    "%Y-%m-%d_%H-%M-%S"
)
LOG_DIR = Path("../../runs/audio_librispeech_conformer").resolve() / RUN_TIMESTAMP
MODEL_DIR = Path("../../models/audio").resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "librispeech_conformer_ctc.zip"

# ── Acoustic Feature Parameters ───────────────────────────────────────────
SAMPLE_RATE = 16000
N_FFT = 512
WIN_LENGTH = 512
HOP_LENGTH = 160
N_MELS = 80

# ── Optimization & Conformer Dimensions ──────────────────────────────────
BATCH_SIZE = 16
EPOCHS = 20
LR = 5e-4
WEIGHT_DECAY = 1e-5
D_MODEL = 144
NUM_HEADS = 4
NUM_LAYERS = 4
CONV_KERNEL_SIZE = 31
NUM_WORKERS = 2 if os.name != "nt" else 0
PIN_MEMORY = device.type == "cuda"

print(
    f"Conformer Config: D_model={D_MODEL}, Heads={NUM_HEADS}, Layers={NUM_LAYERS}, Kernel={CONV_KERNEL_SIZE}"
)

## 6. Character Tokenization and Text Preprocessing

Define a 28-token character-level vocabulary consisting of the CTC blank token $\epsilon$, space character, apostrophe, and English lowercase letters $a\text{--}z$. Implement bidirectional text encoding and decoding routines.

In [ ]:
class CharacterTokenizer:
    """Character-level tokenizer for CTC ASR."""

    def __init__(self):
        self.blank_token = "<blank>"
        self.characters = [self.blank_token, " ", "'"] + [
            chr(c) for c in range(ord("a"), ord("z") + 1)
        ]
        self.char_to_idx = {char: idx for idx, char in enumerate(self.characters)}
        self.idx_to_char = {idx: char for idx, char in enumerate(self.characters)}
        self.blank_idx = 0
        self.vocab_size = len(self.characters)

    def clean_text(self, text: str) -> str:
        text = text.lower().strip()
        text = re.sub(r"[^a-z' ]", "", text)
        text = re.sub(r"\s+", " ", text)
        return text

    def encode(self, text: str) -> list:
        cleaned = self.clean_text(text)
        return [self.char_to_idx[c] for c in cleaned if c in self.char_to_idx]

    def decode(self, tokens: list) -> str:
        return "".join([self.idx_to_char[t] for t in tokens if t != self.blank_idx])


tokenizer = CharacterTokenizer()
print(f"ASR Vocabulary ({tokenizer.vocab_size} tokens): {tokenizer.characters}")

# Test Tokenizer
sample_text = "Hello, LibriSpeech ASR with Conformer!"
encoded_sample = tokenizer.encode(sample_text)
decoded_sample = tokenizer.decode(encoded_sample)
print(
    f"Original: '{sample_text}' -> Encoded: {encoded_sample} -> Decoded: '{decoded_sample}'"
)

## 7. Acoustic Feature Extraction and SpecAugment Transformation

Extract 80-channel log Mel-filterbank spectrograms ($N_{\text{fft}}=512, \text{hop}=160$ for $10\text{ ms}$ frame shift) with training-time SpecAugment frequency and time masking, and implement dynamic batch collation with sequence length tracking.

In [ ]:
class ASRFeatureExtractor(nn.Module):
    """Extracts Log-Mel Filterbank energies with SpecAugment."""

    def __init__(self, is_training=False):
        super().__init__()
        self.mel_transform = T.MelSpectrogram(
            sample_rate=SAMPLE_RATE,
            n_fft=N_FFT,
            win_length=WIN_LENGTH,
            hop_length=HOP_LENGTH,
            n_mels=N_MELS,
            power=2.0,
        )
        self.amplitude_to_db = T.AmplitudeToDB(top_db=80.0)
        self.is_training = is_training
        self.freq_mask = T.FrequencyMasking(freq_mask_param=15)
        self.time_mask = T.TimeMasking(time_mask_param=35)

    def forward(self, waveforms: torch.Tensor) -> torch.Tensor:
        # waveforms: [B, 1, T]
        mel = self.mel_transform(waveforms)
        log_mel = self.amplitude_to_db(mel).squeeze(1)
        if self.is_training and self.training:
            log_mel = self.freq_mask(log_mel)
            log_mel = self.time_mask(log_mel)
        return log_mel.transpose(1, 2)


def collate_asr_batch(batch):
    """Pads audio waveforms and text tokens to batch maximum length."""
    waveforms, targets, input_lengths, target_lengths = [], [], [], []

    for waveform, sample_rate, utterance, speaker_id, chapter_id, utterance_id in batch:
        if sample_rate != SAMPLE_RATE:
            resampler = T.Resample(orig_freq=sample_rate, new_freq=SAMPLE_RATE)
            waveform = resampler(waveform)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        encoded_tokens = torch.tensor(tokenizer.encode(utterance), dtype=torch.long)
        if len(encoded_tokens) == 0:
            continue

        waveforms.append(waveform.squeeze(0))
        targets.append(encoded_tokens)
        input_lengths.append(waveform.shape[1])
        target_lengths.append(len(encoded_tokens))

    # Pad waveforms
    padded_waves = nn.utils.rnn.pad_sequence(waveforms, batch_first=True).unsqueeze(1)
    padded_targets = nn.utils.rnn.pad_sequence(
        targets, batch_first=True, padding_value=tokenizer.blank_idx
    )

    return (
        padded_waves,
        padded_targets,
        torch.tensor(input_lengths, dtype=torch.long),
        torch.tensor(target_lengths, dtype=torch.long),
    )

## 8. Dataset Ingestion and DataLoader Instantiation

Download and cache the LibriSpeech ASR corpus (`dev-clean` split) and construct training and validation DataLoaders with dynamic padding.

In [ ]:
print("Loading LibriSpeech dataset (dev-clean split)...")
raw_librispeech = LIBRISPEECH(root=str(DATA_DIR), url="dev-clean", download=True)
print(f"Total Utterances Loaded: {len(raw_librispeech):,}")

# Split into train and validation subsets
n_total = len(raw_librispeech)
n_train = int(0.85 * n_total)
n_val = n_total - n_train

train_set, val_set = torch.utils.data.random_split(
    raw_librispeech, [n_train, n_val], generator=torch.Generator().manual_seed(42)
)

print(f"Train Utterances: {len(train_set):,} | Val Utterances: {len(val_set):,}")

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_asr_batch,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)

val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_asr_batch,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)

## 9. Exploratory Speech Waveform and Spectrogram Inspection

Inspect sample speech audio waveforms, aligned log Mel-spectrograms, and normalized transcriptions with interactive audio playback widgets.

In [ ]:
sample_wave, sample_sr, sample_text, spk_id, chap_id, utt_id = raw_librispeech[0]
feature_ext = ASRFeatureExtractor(is_training=False)
sample_mel = feature_ext(sample_wave.unsqueeze(0))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))
# Waveform
time_axis = np.linspace(0, sample_wave.shape[1] / sample_sr, sample_wave.shape[1])
ax1.plot(time_axis, sample_wave[0].numpy(), color="#1f77b4", lw=0.8)
ax1.set_title(f"Speech Waveform | Speaker: {spk_id} | Chapter: {chap_id}")
ax1.set_ylabel("Amplitude")
ax1.set_xlabel("Time (s)")
ax1.grid(True, alpha=0.3)

# Mel Spectrogram
im = ax2.imshow(sample_mel[0].T.numpy(), origin="lower", aspect="auto", cmap="viridis")
ax2.set_title(f"Log Mel-Filterbank (80 Bins) | Text: '{sample_text}'")
ax2.set_ylabel("Mel Frequency Bin")
ax2.set_xlabel("Time Frames (10ms)")
fig.colorbar(im, ax=ax2, format="%+2.0f dB")

plt.tight_layout()
plt.show()

print(f"Transcription: '{sample_text}'")
ipd.Audio(sample_wave[0].numpy(), rate=sample_sr)

## 10. Neural Network Architecture Definition

Construct the Conformer ASR architecture:
- **2D Convolution Subsampling**: $4\times$ temporal subsampling stem using 2D depthwise-separable convolutions and linear projection to $d_{\text{model}}$.
- **Conformer Encoder Blocks**: Macaron-style half-step Feed-Forward module, Multi-Head Self-Attention (MHSA) with relative positional embeddings, 1D Depthwise Convolution ($k=31$) with Gated Linear Units (GLU), and final half-step Feed-Forward module.
- **CTC Linear Head**: Projects encoder frame representations to character vocabulary log-probabilities.

In [ ]:
class Conv2dSubsampling(nn.Module):
    """4x Conv2D Subsampling Stem."""

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, out_channels, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
        )
        # Compute output feature dimension after 2 stride-2 convolutions
        subsampled_freq = (((in_channels + 2 * 1 - 3) // 2 + 1) + 2 * 1 - 3) // 2 + 1
        self.proj = nn.Linear(out_channels * subsampled_freq, out_channels)

    def forward(self, x: torch.Tensor, lengths: torch.Tensor):
        # x: [B, T, F] -> [B, 1, T, F]
        x = x.unsqueeze(1)
        x = self.conv(x)
        B, C, T_sub, F_sub = x.shape
        x = x.permute(0, 2, 1, 3).reshape(B, T_sub, C * F_sub)
        x = self.proj(x)
        # Update output frame lengths: (lengths // 4)
        out_lengths = ((lengths + 1) // 2 + 1) // 2
        return x, out_lengths


class ConformerConvModule(nn.Module):
    """Conformer Convolution Module with Gated Linear Units and Depthwise Conv."""

    def __init__(self, d_model: int, kernel_size: int = 31, dropout: float = 0.1):
        super().__init__()
        self.layer_norm = nn.LayerNorm(d_model)
        self.pw_conv1 = nn.Conv1d(d_model, 2 * d_model, kernel_size=1)
        self.glu = nn.GLU(dim=1)
        self.dw_conv = nn.Conv1d(
            d_model,
            d_model,
            kernel_size=kernel_size,
            padding=(kernel_size - 1) // 2,
            groups=d_model,
        )
        self.batch_norm = nn.BatchNorm1d(d_model)
        self.activation = nn.SiLU()
        self.pw_conv2 = nn.Conv1d(d_model, d_model, kernel_size=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, D]
        residual = x
        x = self.layer_norm(x)
        x = x.transpose(1, 2)
        x = self.pw_conv1(x)
        x = self.glu(x)
        x = self.dw_conv(x)
        x = self.batch_norm(x)
        x = self.activation(x)
        x = self.pw_conv2(x)
        x = self.dropout(x)
        x = x.transpose(1, 2)
        return residual + x


class ConformerFeedForward(nn.Module):
    """Conformer Macaron-style Feed-Forward Network."""

    def __init__(self, d_model: int, expansion: int = 4, dropout: float = 0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.linear1 = nn.Linear(d_model, d_model * expansion)
        self.act = nn.SiLU()
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_model * expansion, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = self.norm(x)
        x = self.linear2(self.dropout(self.act(self.linear1(x))))
        return residual + 0.5 * self.dropout(x)


class ConformerBlock(nn.Module):
    """Single Conformer Block with Macaron-FFN, Self-Attention, and Conv Module."""

    def __init__(
        self,
        d_model: int,
        num_heads: int = 4,
        kernel_size: int = 31,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.ffn1 = ConformerFeedForward(d_model, expansion=4, dropout=dropout)
        self.attn_norm = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(
            d_model, num_heads, dropout=dropout, batch_first=True
        )
        self.conv_module = ConformerConvModule(
            d_model, kernel_size=kernel_size, dropout=dropout
        )
        self.ffn2 = ConformerFeedForward(d_model, expansion=4, dropout=dropout)
        self.final_norm = nn.LayerNorm(d_model)

    def forward(
        self, x: torch.Tensor, key_padding_mask: torch.Tensor = None
    ) -> torch.Tensor:
        x = self.ffn1(x)

        # Self-Attention with pre-LayerNorm
        norm_x = self.attn_norm(x)
        attn_out, _ = self.self_attn(
            norm_x, norm_x, norm_x, key_padding_mask=key_padding_mask
        )
        x = x + attn_out

        x = self.conv_module(x)
        x = self.ffn2(x)
        x = self.final_norm(x)
        return x


class ConformerASR(nn.Module):
    """End-to-End Conformer ASR Model with Feature Extraction and CTC Projection."""

    def __init__(
        self,
        vocab_size: int,
        n_mels: int = 80,
        d_model: int = 144,
        num_heads: int = 4,
        num_layers: int = 4,
        conv_kernel_size: int = 31,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.feature_extractor = ASRFeatureExtractor(is_training=True)
        self.subsampling = Conv2dSubsampling(n_mels, d_model)

        self.layers = nn.ModuleList(
            [
                ConformerBlock(
                    d_model,
                    num_heads=num_heads,
                    kernel_size=conv_kernel_size,
                    dropout=dropout,
                )
                for _ in range(num_layers)
            ]
        )

        self.ctc_head = nn.Linear(d_model, vocab_size)

    def forward(self, waveforms: torch.Tensor, input_lengths: torch.Tensor):
        # waveforms: [B, 1, Samples]
        mels = self.feature_extractor(waveforms)
        # Frame lengths before subsampling: (samples // hop_length) + 1
        frame_lengths = (input_lengths // HOP_LENGTH) + 1

        x, out_lengths = self.subsampling(mels, frame_lengths)

        for layer in self.layers:
            x = layer(x)

        logits = self.ctc_head(x)
        log_probs = F.log_softmax(logits, dim=-1)
        return log_probs, out_lengths

## 11. Model Instantiation and Parameter Summary

Instantiate the Conformer ASR model on the compute device, audit total trainable parameters, and verify forward pass shape consistency.

In [ ]:
asr_model = ConformerASR(
    vocab_size=tokenizer.vocab_size,
    n_mels=N_MELS,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    conv_kernel_size=CONV_KERNEL_SIZE,
).to(device)

total_params = sum(p.numel() for p in asr_model.parameters())
print(f"Conformer ASR Architecture Summary:")
print(f"Total Parameters: {total_params:,} ({total_params * 4 / (1024**2):.2f} MB)")

# Test dummy batch
dummy_waves = torch.randn(2, 1, 32000, device=device)
dummy_lens = torch.tensor([32000, 24000], device=device)
with torch.no_grad():
    dummy_log_probs, dummy_out_lens = asr_model(dummy_waves, dummy_lens)

print(
    f"Output Log-Probs Shape: {dummy_log_probs.shape} (B, Subsampled_Frames, Vocab_Size)"
)
print(f"Subsampled Sequence Lengths: {dummy_out_lens.cpu().numpy()}")

## 12. CTC Decoding and Error Rate Metrics

Implement frame-wise greedy best-path CTC decoding and character/word error rate (CER/WER) evaluation using Levenshtein edit distance:
$$\text{CER} = \frac{S + D + I}{N}$$

In [ ]:
def ctc_greedy_decode(log_probs: torch.Tensor, out_lengths: torch.Tensor) -> list:
    """Collapses repeat tokens and removes CTC blank token (index 0)."""
    preds = log_probs.argmax(dim=-1).cpu().numpy()
    decoded_strings = []

    for i, seq in enumerate(preds):
        length = out_lengths[i].item()
        collapsed = []
        prev_token = None
        for token in seq[:length]:
            if token != prev_token:
                if token != tokenizer.blank_idx:
                    collapsed.append(token)
                prev_token = token
        decoded_strings.append(tokenizer.decode(collapsed))

    return decoded_strings


def compute_levenshtein(seq1: list, seq2: list) -> int:
    """Dynamic programming computation of minimum edit distance."""
    d = np.zeros((len(seq1) + 1, len(seq2) + 1), dtype=int)
    for i in range(len(seq1) + 1):
        d[i, 0] = i
    for j in range(len(seq2) + 1):
        d[0, j] = j
    for i in range(1, len(seq1) + 1):
        for j in range(1, len(seq2) + 1):
            if seq1[i - 1] == seq2[j - 1]:
                d[i, j] = d[i - 1, j - 1]
            else:
                d[i, j] = min(d[i - 1, j] + 1, d[i, j - 1] + 1, d[i - 1, j - 1] + 1)
    return d[len(seq1), len(seq2)]


def compute_wer_cer(predictions: list, references: list):
    """Calculates Word Error Rate and Character Error Rate."""
    total_char_dist, total_chars = 0, 0
    total_word_dist, total_words = 0, 0

    for pred, ref in zip(predictions, references):
        pred_clean = tokenizer.clean_text(pred)
        ref_clean = tokenizer.clean_text(ref)

        # Character Error Rate
        total_char_dist += compute_levenshtein(list(pred_clean), list(ref_clean))
        total_chars += max(1, len(ref_clean))

        # Word Error Rate
        pred_words = pred_clean.split()
        ref_words = ref_clean.split()
        total_word_dist += compute_levenshtein(pred_words, ref_words)
        total_words += max(1, len(ref_words))

    cer = (total_char_dist / total_chars) * 100.0
    wer = (total_word_dist / total_words) * 100.0
    return cer, wer

## 13. Loss Function, Optimizer, and Training Loop Execution

Configure Connectionist Temporal Classification (CTC) loss, `AdamW` optimizer with linear warmup and cosine decay, gradient norm clipping ($5.0$), validation CER tracking, and checkpoint saving.

In [ ]:
ctc_loss_fn = nn.CTCLoss(blank=tokenizer.blank_idx, zero_infinity=True)
optimizer = torch.optim.AdamW(asr_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6
)

writer = SummaryWriter(log_dir=str(LOG_DIR))
scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None


def evaluate_asr(model, dataloader, device):
    model.eval()
    total_loss, total_samples = 0.0, 0
    all_preds, all_refs = [], []

    with torch.no_grad():
        for waveforms, targets, input_lens, target_lens in dataloader:
            waveforms, targets = waveforms.to(device), targets.to(device)
            input_lens, target_lens = input_lens.to(device), target_lens.to(device)

            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                log_probs, out_lens = model(waveforms, input_lens)
                loss = ctc_loss_fn(
                    log_probs.transpose(0, 1), targets, out_lens, target_lens
                )

            total_loss += loss.item() * waveforms.size(0)
            total_samples += waveforms.size(0)

            preds = ctc_greedy_decode(log_probs, out_lens)
            refs = [
                tokenizer.decode(t.cpu().numpy()[: target_lens[i].item()])
                for i, t in enumerate(targets)
            ]
            all_preds.extend(preds)
            all_refs.extend(refs)

    cer, wer = compute_wer_cer(all_preds, all_refs)
    return total_loss / total_samples, cer, wer, all_preds, all_refs


best_val_cer = 100.0
print(f"Starting Conformer ASR training for {EPOCHS} epochs on {device_name}...")

for epoch in range(1, EPOCHS + 1):
    asr_model.train()
    train_loss, train_samples = 0.0, 0

    for waveforms, targets, input_lens, target_lens in train_loader:
        waveforms, targets = waveforms.to(device), targets.to(device)
        input_lens, target_lens = input_lens.to(device), target_lens.to(device)
        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:
            with torch.amp.autocast("cuda"):
                log_probs, out_lens = asr_model(waveforms, input_lens)
                loss = ctc_loss_fn(
                    log_probs.transpose(0, 1), targets, out_lens, target_lens
                )
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(asr_model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            log_probs, out_lens = asr_model(waveforms, input_lens)
            loss = ctc_loss_fn(
                log_probs.transpose(0, 1), targets, out_lens, target_lens
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(asr_model.parameters(), max_norm=5.0)
            if is_xla:
                xm.optimizer_step(optimizer)
            else:
                optimizer.step()

        train_loss += loss.item() * waveforms.size(0)
        train_samples += waveforms.size(0)

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    epoch_train_loss = train_loss / train_samples
    epoch_val_loss, val_cer, val_wer, sample_preds, sample_refs = evaluate_asr(
        asr_model, val_loader, device
    )

    writer.add_scalar("Loss/Train_CTC", epoch_train_loss, epoch)
    writer.add_scalar("Loss/Val_CTC", epoch_val_loss, epoch)
    writer.add_scalar("Metrics/CER", val_cer, epoch)
    writer.add_scalar("Metrics/WER", val_wer, epoch)
    writer.flush()

    print(
        f"Epoch [{epoch:02d}/{EPOCHS:02d}] "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | Val CER: {val_cer:.2f}% | Val WER: {val_wer:.2f}% | "
        f"LR: {current_lr:.2e}"
    )

    if val_cer < best_val_cer:
        best_val_cer = val_cer
        save_model_zip(
            model=asr_model,
            metadata={
                "epoch": epoch,
                "cer": best_val_cer,
                "wer": val_wer,
                "vocab": tokenizer.characters,
            },
            save_path=MODEL_PATH,
            is_xla=is_xla,
        )
        print(
            f"  --> Saved new best checkpoint to {MODEL_PATH} (CER: {best_val_cer:.2f}%, sample: '{sample_preds[0]}')"
        )

writer.close()
print("Training completed.")

## 14. Interactive Speech Recognition Inference

Demonstrate end-to-end speech recognition on audio samples, transcribing speech waveforms into text and comparing predictions against ground-truth transcripts.

In [ ]:
def transcribe_speech(waveform: torch.Tensor, sample_rate: int):
    """Runs end-to-end ASR inference on a waveform."""
    asr_model.eval()
    if sample_rate != SAMPLE_RATE:
        resampler = T.Resample(orig_freq=sample_rate, new_freq=SAMPLE_RATE)
        waveform = resampler(waveform)
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    input_wave = waveform.unsqueeze(0).to(device)
    input_len = torch.tensor([waveform.shape[1]], device=device)

    with torch.no_grad():
        log_probs, out_lens = asr_model(input_wave, input_len)
        hypothesis = ctc_greedy_decode(log_probs, out_lens)[0]

    return hypothesis, log_probs[0].cpu().numpy()


# Test sample from validation set
val_idx = random.randint(0, len(val_set) - 1)
test_wave, test_sr, test_ref, _, _, _ = val_set[val_idx]
pred_text, log_probs_matrix = transcribe_speech(test_wave, test_sr)

print("=" * 60)
print(f"Reference Transcript: '{test_ref.lower()}'")
print(f"Predicted Transcript: '{pred_text}'")
print("=" * 60)

# Plot Token Posterior Probabilities across time
plt.figure(figsize=(14, 4))
plt.imshow(log_probs_matrix.T, origin="lower", aspect="auto", cmap="viridis")
plt.title(f"CTC Log-Posterior Distribution across Time Frames")
plt.xlabel("Subsampled Time Frames")
plt.ylabel("Vocabulary Tokens")
plt.colorbar(label="Log Probability")
plt.tight_layout()
plt.show()

# Interactive Audio Playback
ipd.Audio(test_wave[0].numpy(), rate=test_sr)